# Gold Layer: Analytics-ready datasets powering business decisions (KPIs, rankings, trends)

In [0]:
# Using Silver layer tables as input for Gold processing

customers_df = spark.table("silver_customers")
orders_df = spark.table("silver_orders")
order_items_df = spark.table("silver_order_item")
sellers_df = spark.table("silver_sellers")
reviews_df = spark.table("silver_reviews")
products_df = spark.table("silver_products")
payments_df = spark.table("silver_payments")
geolocation_df = spark.table("silver_geolocation")
full_order_df = spark.table("silver_full_order")
order_details_df = spark.table("order_details")

In [0]:
from pyspark.sql.functions import *

In [0]:
# Total value of order
order_total_value = order_details_df.groupBy("order_id")\
    .agg(sum("payment_value").alias("total_order_value"))

order_total_value.show()

# save table 
order_total_value.write.mode("overwrite").saveAsTable("order_total_value")

In [0]:
# Total Revenue Per Seller
total_revenue = full_order_df.groupBy("seller_id").agg(sum("price").alias("total_revenue")).orderBy("total_revenue",ascending=False)

total_revenue.show(5)

# save table 
total_revenue.write.mode("overwrite").saveAsTable("total_revenue")

In [0]:
# Total Orders per customer

total_orders_per_customer_df = full_order_df.groupBy("customer_id").agg(count("order_id").alias("Total Orders"))
total_orders_per_customer_df.show(5)

# save table 
total_orders_per_customer_df.write.mode("overwrite").saveAsTable("total_orders_per_customer_df")

In [0]:
# Average review score per seller

review_score_df = full_order_df.groupBy("seller_id").agg(avg("review_score").alias("Avg Review"))
review_score_df.show(5)

# save table 
review_score_df.write.mode("overwrite").saveAsTable("review_score_df")


In [0]:
# Most sold products (top 10)

top_products_df = full_order_df.groupBy("product_id")\
              .agg(count("order_item_id").alias("Top product"))\
              .orderBy("Top product",ascending=False)

top_products_df.show(10)

# save table 
top_products_df.write.mode("overwrite").saveAsTable("top_products_df")

In [0]:
# Top customers by spending

top_customer_df = full_order_df.groupBy("customer_id").agg(sum("price").alias("total spend")).orderBy(desc("total spend"))
top_customer_df.show(5)

# save table 
top_customer_df.write.mode("overwrite").saveAsTable("top_customer_df")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("seller_id").orderBy(desc("price"))    

# Rank top selling products per seller
top_rank_products_df = full_order_df.withColumn("rank",rank().over(window_spec)).filter(col("rank")<=5)

top_rank_products_df.show(5)

# save table 
top_rank_products_df.write.mode("overwrite").saveAsTable("top_rank_products_df")

In [0]:
# Dense rank for seller based on revenue

seller_revenue_df = full_order_df.groupBy("seller_id").agg(sum("price").alias("total_revenue"))
seller_dense_rank_window = Window.orderBy(desc("total_revenue"))
seller_rank_revenue = seller_revenue_df.withColumn("dense_rank",dense_rank().over(seller_dense_rank_window))


seller_rank_revenue.show()

# save table 
seller_rank_revenue.write.mode("overwrite").saveAsTable("seller_rank_revenue")

In [0]:
# Total revenue and Average Order Value (AOV) per customer

customer_spend_df = full_order_df.groupBy("customer_id")\
    .agg(
        count("order_id").alias("Total_orders"),
        sum("price").alias("Total_spent"),
        round(avg("price"),2).alias("AOV")
    )\
        .orderBy(desc("Total_spent"))

customer_spend_df.show()

customer_spend_df.write.mode("overwrite").saveAsTable("Revenue_AOV")

In [0]:
# Seller performance metrics (Revenue, Average review, Order count)

seller_performance_df = full_order_df.groupBy("seller_id")\
.agg(
    count("order_id").alias("Total_orders"),
    sum("price").alias("Total_revenue"),
    round(avg("review_score"),2).alias("Avg_review"),
    round(stddev("price"),2).alias("Price_variability")
)\
    .orderBy(desc("Total_revenue"))

seller_performance_df.show()    


In [0]:
# Product Popularity Metrics

product_metrics_df = full_order_df.groupBy("product_id")\
    .agg(
        count("order_id").alias("Total_sales"),
        sum("price").alias("Total_revenue"),
        round(avg("price"),2).alias("Avg_price"),
        round(stddev("price"),2).alias("Price_volatility"),
        collect_set("seller_id").alias("Unique_seller")
    )\
        .orderBy("Total_sales")

product_metrics_df.show()     

product_metrics_df.write.mode("overwrite").saveAsTable("product_metrics")
